# Full MEA Pipeline

Runs all six analysis stages end-to-end for every well in the dataset.

```
preprocessing → sorting → auto_merge → analyzer → auto_curation → burst_detection
```

| Stage | Task | Typical runtime |
|---|---|---|
| 1 | `preprocessing` | ~1–5 min / well |
| 2 | `sorting` | ~10–60 min / well (GPU-dependent) |
| 3 | `auto_merge` | seconds (disabled) / ~5 min (enabled) |
| 4 | `analyzer` | ~5–20 min / well |
| 5 | `auto_curation` | < 1 min / well |
| 6 | `burst_detection` | < 1 min / well |

**Configuration:** edit `pipeline_config.json` (generated on first run) to set paths,
sorting parameters, curation thresholds, and merge options.

**Re-runs are safe:** the pipeline cache skips completed wells automatically.
Changing any parameter in `pipeline_config.json` invalidates the affected task
and all downstream tasks for wells that have that config cached.

In [ ]:
import sys
import logging
from pathlib import Path

import pandas as pd

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager, PipelineRunOptions, run_pipeline_session
from pipeline_manager.task_record import TaskStatus
from pipeline_tasks import (
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
    PlateViewerTask,
)
from config_manager import ConfigManager

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

## Task chain

Inspect the registered task order and dependency graph before doing anything else.

In [ ]:
WELL_TASK_CLASSES = [
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
]

PLATE_TASK_CLASSES = [PlateViewerTask]
ALL_TASK_CLASSES = WELL_TASK_CLASSES + PLATE_TASK_CLASSES

TASK_ORDER = [cls.task_name for cls in WELL_TASK_CLASSES]

print("Pipeline execution order:")
for i, cls in enumerate(ALL_TASK_CLASSES, 1):
    deps = " → ".join(cls.dependencies) if cls.dependencies else "(none)"
    print(f"  {i}. {cls.task_name:<20s}  depends on: {deps}")

## Config

On the **first run** this cell writes `pipeline_config.json` with all task defaults and
stops — edit it to set `data_root`, `analysis_root`, and any parameters you want to
override, then re-run.

Key parameters to review before the first full run:

| Section | Key | What to set |
|---|---|---|
| `global` | `data_root` | Sample/root folder that contains date directories, e.g. `/mnt/benshalom-nas/raw_data/rbs_maxtwo_desktop/harddisk20tbV2/Sadegh_Lab/CX138` |
| `global` | `analysis_root` | Where all outputs are written |
| `sorting` | `sorter` | Sorter name (`kilosort4`) |
| `auto_merge` | `enabled` | `false` to skip merging |
| `auto_curation` | `enabled` | `false` to keep all units |
| `auto_curation` | `*_min` / `*_max` | Curation thresholds |

In [ ]:
CONFIG_FILE = Path("../config/default_tasks_params copy.json")

cm = ConfigManager()

# Path override
# cm.set_global("data_root", "/path/to/raw/data")
# cm.set_global("analysis_root", "/path/to/analysis")

for cls in ALL_TASK_CLASSES:
    cm.register_task(cls)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it — set data_root, analysis_root, and any parameters — "
        "then re-run this cell."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print()
for cls in ALL_TASK_CLASSES:
    params = cm.get_task_params(cls.task_name)
    print(f"  [{cls.task_name}]  {params}")


## 1. Scan recordings

In [ ]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_SCAN_TYPE = "Network"
TARGET_RUN_ID    = "000029"

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_recording_by([("scan_type", "==", TARGET_SCAN_TYPE),
                                            ("run_id", "==", TARGET_RUN_ID)])

if not recordings:
    raise RuntimeError(
        "No recordings matched the notebook filters.\n"
        f"  data_root: {DATA_ROOT}\n"
        f"  filters: scan_type == {TARGET_SCAN_TYPE!r}, run_id == {TARGET_RUN_ID!r}\n"
        "Expected data_root to be a sample/root folder containing date directories, "
        "for example: .../Sadegh_Lab/CX138."
    )

print(f"Found {len(recordings)} Network recording(s)")

pd.DataFrame([
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "scan_type":     r.scan_type,
        "run_id":        r.run_id,
        "n_wells":       len(r.wells),
        "size_GB":       round(r.file_size / 1e9, 2),
    }
    for r in recordings
])

## 2. Register all tasks and add wells

In [ ]:
pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

for cls in WELL_TASK_CLASSES:
    try:
        pipeline_mgr.register_task(cls)
        print(f"  registered: {cls.task_name!r}")
    except ValueError as e:
        print(f"  already registered: {cls.task_name!r} ({e})")

n_added   = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Recordings skipped: {n_skipped}")

## 3. Pipeline status overview

The matrix below shows the current status of every (well × task) combination.
Run this cell at any time to check progress.

In [ ]:
_STATUS_SYMBOL = {
    str(TaskStatus.NOT_RUN):  "—",
    str(TaskStatus.RUNNING):  "⏳",
    str(TaskStatus.COMPLETE): "✓",
    str(TaskStatus.FAILED):   "✗",
}


def _pipeline_matrix(mgr: PipelineManager) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        if "/" not in entry.well_id:
            continue
        rec_name, well_id = entry.well_id.split("/", 1)
        row: dict = {
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
        }
        for task_name in TASK_ORDER:
            t = entry.tasks.get(task_name)
            status_str = str(t.status) if t else str(TaskStatus.NOT_RUN)
            row[task_name] = _STATUS_SYMBOL.get(status_str, status_str)
        rows.append(row)
    columns = ["recording_key", "rec_name", "well_id", *TASK_ORDER]
    return pd.DataFrame(rows, columns=columns)


df_matrix = _pipeline_matrix(pipeline_mgr)
print("Legend:  — not run   ⏳ running   ✓ complete   ✗ failed")
df_matrix


## 4. Refresh controls

Run this optional cell before the main pipeline loop when you want to re-run selected work.
`PipelineManager.refresh()` applies to cached per-well tasks. `plate_viewer` runs once per
recording outside the per-well task cache, so refreshing it means regenerating the HTML output.

In [ ]:
# Optional refresh examples. Uncomment one before running the pipeline.

# One task across all wells, including downstream dependents:
# pipeline_mgr.refresh("sorting")

# No refresh call is needed for plate_viewer. Section 4 rewrites plate_viewer.html every run.

# One task for one recording, including downstream dependents:
# pipeline_mgr.refresh("sorting", recording_key="SampleA/240415/PlateX/Network/001")

# One task for one well, including downstream dependents:
# pipeline_mgr.refresh(
#     "sorting",
#     recording_key="SampleA/240415/PlateX/Network/001",
#     well_id="rec0000/well000",
# )


## 4. Run full pipeline

`run_pipeline_session()` wraps the per-well task loop, status updates, retry handling,
stop-after behavior, and plate viewer generation.

**To selectively retry failed tasks** add their names to `retry_failed_tasks`.

**To stop after a specific stage** set `stop_after_task`, for example `"sorting"`.

This notebook is set to regenerate only `plate_viewer.html` every time this cell runs.
Set `plate_viewer_only=False` when you want to run pending well-level tasks first.


In [ ]:
options = PipelineRunOptions(
    retry_failed_tasks=set(),   # e.g. {"analyzer", "auto_curation"}
    stop_after_task=None,       # e.g. "sorting" to pause after sorting
    plate_viewer_only=True,     # rewrites plate_viewer.html every run
    plate_recording_keys=set(), # empty means all currently loaded recordings
    run_plate_viewer=True,
)

result = run_pipeline_session(
    pipeline_mgr=pipeline_mgr,
    dataset_mgr=dataset_mgr,
    config_provider=cm,
    recordings=recordings,
    well_task_classes=WELL_TASK_CLASSES,
    plate_task_class=PlateViewerTask,
    options=options,
)


## 5. Final status

In [ ]:
df_final = _pipeline_matrix(pipeline_mgr)

if df_final.empty:
    print("No pipeline entries are registered for the current session.")
else:
    print("=== Per-task summary ===")
    for task_name in TASK_ORDER:
        col = df_final[task_name]
        counts = col.value_counts().to_dict()
        print(f"  {task_name:<20s}  {counts}")

failed_mask = (
    (df_final[TASK_ORDER] == "\u2717").any(axis=1)
    if not df_final.empty
    else pd.Series(dtype=bool)
)
if not df_final.empty and failed_mask.any():
    print("\n=== Failed wells ===")
    for _, row in df_final[failed_mask].iterrows():
        failed_tasks = [t for t in TASK_ORDER if row[t] == "\u2717"]
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    failed stages: {failed_tasks}")

if globals().get("result") and result.plate_outputs:
    print("\n=== Plate viewer outputs ===")
    for recording_key, output_path in result.plate_outputs.items():
        print(f"  {recording_key}: {output_path}")

print("\nLegend:  \u2014 not run   \u23f3 running   \u2713 complete   \u2717 failed")
df_final

## 6. Curation summary

Reads `quality_metrics.pkl` from every completed `auto_curation` well
and produces an across-wells summary table.

In [ ]:
summary_rows = []

for entry in pipeline_mgr.entries:
    if "/" not in entry.well_id:
        continue
    curation_task = entry.tasks.get(AutoCurationTask.task_name)
    if (
        not curation_task
        or curation_task.status != TaskStatus.COMPLETE
        or not curation_task.output_path
    ):
        continue

    qm_path = Path(curation_task.output_path) / "quality_metrics.pkl"
    if not qm_path.exists():
        continue

    rec_name, well_id = entry.well_id.split("/", 1)
    qm = pd.read_pickle(qm_path)

    curated = qm[qm["curated"]]
    summary_rows.append({
        "recording_key": entry.recording_key,
        "rec_name":      rec_name,
        "well_id":       well_id,
        "n_total":        len(qm),
        "n_curated":      len(curated),
        "pct_kept":       round(100 * len(curated) / len(qm), 1) if len(qm) else 0,
        "median_fr_hz":   round(curated["firing_rate"].median(), 3)
                          if "firing_rate" in curated.columns and len(curated) else None,
        "median_amp_uv":  round(curated["amplitude_median"].median(), 1)
                          if "amplitude_median" in curated.columns and len(curated) else None,
    })

if summary_rows:
    df_curation = pd.DataFrame(summary_rows)
    print(f"Wells with curation results: {len(df_curation)}")
    print(f"Total curated units: {df_curation['n_curated'].sum()}  /  {df_curation['n_total'].sum()}")
    df_curation
else:
    print("No curation results yet — run the pipeline first.")
